# Inversion evaluation

Compare camp-A inverters on held-out FlatVel_A maps through the shared
`physics_informed_flow_map.inversion` harness: an `Evaluator` scores any
`InversionModule` on the same held-out targets and Deepwave operator. Here the
**flow-map (0002)** and **diffusion (0003)** priors, each tilted/DPS-guided vs its
unguided prior-only control.

Metrics are reported under three selection rules — `oracle` (lowest MAE, needs the
truth), `gt_free` (lowest data misfit, what a real inversion must use), and
`posterior_mean` — plus `n_solves` (matched-cost PDE-solve count).

In [ ]:
import os
from pathlib import Path

import torch
from diffusers import DDPMScheduler

from physics_informed_flow_map.baselines import build_denoiser
from physics_informed_flow_map.flow_matching.models import DiTModelConfig, build_model
from physics_informed_flow_map.inversion import (
    DiffusionDPSModule,
    Evaluator,
    FlowTiltModule,
)

# Hop to the repo root so the gitignored runs/ and data/ resolve whether this is run from
# notebooks/ (nbconvert's default cwd) or the repo root (interactive jupyter).
_root = Path.cwd()
while not (_root / ".git").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# FlatVel_A prior checkpoints (gitignored runs/ — update to your latest). All four priors share
# the same DiT backbone except the UNet diffusion, so flow-matching vs flow-map isolates the
# flow-map loss, and diffusion-UNet vs diffusion-DiT isolates the backbone.
FLOW_CKPT = "runs/0001_flow_matching/2026-06-26T23-40-26Z/checkpoints/step_99_ema.pt"
FLOWMAP_CKPT = "runs/0002_flow_map/2026-06-26T20-57-58Z/checkpoints/step_99_ema.pt"
DIFF_UNET_CKPT = "runs/0003_baselines/2026-06-26T11-22-31Z/checkpoints/step_99_ema.pt"
DIFF_DIT_CKPT = "runs/0003_baselines/2026-06-27T00-21-12Z/checkpoints/step_99_ema.pt"
N_TARGETS = 4  # held-out FlatVel_A val maps to average over
STEPS, N_SAMPLES = 200, 4
RUN_SWEEP = False  # the guidance sweep is many extra wave solves; off by default

In [ ]:
# Load the priors (I/O at the edge; the modules just consume them).
def load_flow(ckpt):
    m = build_model(
        (1, 64, 64), None, DiTModelConfig(hidden=256, depth=6, num_heads=8, patch_size=4)
    ).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False)["model"])
    return m.eval()


def load_denoiser(kind, ckpt):
    d = build_denoiser(kind, sample_size=64, channels=1).to(device)
    d.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False)["model"])
    return d.eval()


flow_prior = load_flow(FLOW_CKPT)        # 0001 flow-matching
flowmap_prior = load_flow(FLOWMAP_CKPT)  # 0002 flow-map
unet_denoiser = load_denoiser("unet", DIFF_UNET_CKPT)  # 0003 diffusion, UNet backbone
dit_denoiser = load_denoiser("dit", DIFF_DIT_CKPT)     # 0003 diffusion, DiT backbone (matches flow)
scheduler = DDPMScheduler(num_train_timesteps=1000)

ev = Evaluator.from_openfwi(["FlatVel_A"], N_TARGETS, device=device)
print(f"{len(ev.targets)} held-out targets")

In [ ]:
# All four priors on the same held-out maps, guided vs unguided prior-only control. Clear names
# (the modules' auto-names collide) so the table reads as flow-matching / flow-map / diff-unet /
# diff-dit. flow tilt @ g=1.0, diffusion DPS @ g=0.3.
def named(module, name):
    module.name = name
    return module


def flow(prior, g):
    return FlowTiltModule(prior, guidance=g, steps=STEPS, n_samples=N_SAMPLES, device=device)


def dps(denoiser, g):
    return DiffusionDPSModule(denoiser, scheduler, guidance=g, steps=STEPS, n_samples=N_SAMPLES, device=device)


modules = [
    named(flow(flow_prior, 1.0), "fm·g1"),
    named(flow(flow_prior, 0.0), "fm·g0"),
    named(flow(flowmap_prior, 1.0), "flowmap·g1"),
    named(flow(flowmap_prior, 0.0), "flowmap·g0"),
    named(dps(unet_denoiser, 0.3), "diff_unet·g0.3"),
    named(dps(unet_denoiser, 0.0), "diff_unet·g0"),
    named(dps(dit_denoiser, 0.3), "diff_dit·g0.3"),
    named(dps(dit_denoiser, 0.0), "diff_dit·g0"),
]

for m in modules:
    torch.manual_seed(0)  # same posterior noise across modules for a fair compare
    print(ev.evaluate(m), "\n")  # InversionStats.__str__ prints the metric table

## Guidance sweep

MAE (GT-free pick) vs guidance strength for each method on the same held-out set.

In [ ]:
import matplotlib.pyplot as plt

GUIDANCES = [0.3, 1.0, 3.0, 10.0]

if RUN_SWEEP:  # expensive: len(GUIDANCES) x 2 methods x N_TARGETS guided inversions

    def sweep(make_module):
        out = []
        for g in GUIDANCES:
            torch.manual_seed(0)
            out.append(ev.evaluate(make_module(g)).agg["mae_gt_free_mean"])
        return out

    flow_mae = sweep(lambda g: FlowTiltModule(flowmap_prior, guidance=g, steps=STEPS, n_samples=N_SAMPLES, device=device))
    diff_mae = sweep(lambda g: DiffusionDPSModule(denoiser, scheduler, guidance=g, steps=STEPS, n_samples=N_SAMPLES, device=device))

    plt.figure(figsize=(5, 3.4))
    plt.plot(GUIDANCES, flow_mae, "o-", label="flow-map tilt")
    plt.plot(GUIDANCES, diff_mae, "s-", label="diffusion DPS")
    plt.xscale("log")
    plt.xlabel("guidance strength")
    plt.ylabel("MAE (m/s), GT-free pick")
    plt.legend()
    plt.tight_layout()
    plt.show()